# MLOps Overview - Solutions

> 📘 **Python Mastery** · Module 18 — MLOps · Lesson 1/6

## Solution 1: Understanding MLOps

**Answer:**

Traditional DevOps is insufficient for ML because a trained model is not just code—it's a function of code, data, and learned weights. While DevOps tracks code changes, ML systems also require versioning data snapshots and model artifacts. Additionally, ML systems decay silently as the statistical distribution of real-world data shifts away from training data, requiring continuous monitoring of inputs and outputs beyond traditional error logs.

The three core artifacts are:
1. **Code** (algorithms, preprocessing)
2. **Data** (training datasets, versions)
3. **Learned weights/models** (trained artifacts)

## Solution 2: Drift Detective

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

np.random.seed(42)

def generate_fraud_data(n_samples, fraud_ratio, 
                       amount_mean_legit, amount_mean_fraud,
                       time_mean_fraud, merchant_fraud_prob):
    """Generate synthetic fraud dataset with configurable patterns."""
    n_fraud = int(n_samples * fraud_ratio)
    n_legit = n_samples - n_fraud
    
    # Legitimate transactions
    legit_amount = np.abs(np.random.normal(amount_mean_legit, 50, n_legit))
    legit_time = np.random.uniform(6, 22, n_legit)  # daytime
    legit_merchant = np.random.choice([0, 1, 2], n_legit, p=[0.5, 0.3, 0.2])
    
    # Fraudulent transactions
    fraud_amount = np.abs(np.random.normal(amount_mean_fraud, 80, n_fraud))
    fraud_time = np.random.normal(time_mean_fraud, 3, n_fraud) % 24
    fraud_merchant = np.random.choice([0, 1, 2], n_fraud, p=merchant_fraud_prob)
    
    X = np.vstack([
        np.column_stack([legit_amount, legit_time, legit_merchant]),
        np.column_stack([fraud_amount, fraud_time, fraud_merchant])
    ])
    y = np.concatenate([np.zeros(n_legit), np.ones(n_fraud)])
    
    return X, y

# Month 1: Original patterns
X_train, y_train = generate_fraud_data(
    n_samples=2000, fraud_ratio=0.05,
    amount_mean_legit=100, amount_mean_fraud=400,
    time_mean_fraud=2.0,  # 2 AM fraud spike
    merchant_fraud_prob=[0.2, 0.3, 0.5]  # category 2 most risky
)

X_test_m1, y_test_m1 = generate_fraud_data(
    n_samples=500, fraud_ratio=0.05,
    amount_mean_legit=100, amount_mean_fraud=400,
    time_mean_fraud=2.0,
    merchant_fraud_prob=[0.2, 0.3, 0.5]
)

# Train baseline model
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
acc_baseline = model.score(X_test_m1, y_test_m1)

print(f"Month 1 Accuracy: {acc_baseline:.3f}")

# Month 6: Both drift types
# DATA DRIFT: E-commerce boom increases legitimate high-value transactions
# CONCEPT DRIFT: Fraudsters shift to daytime + different merchant categories
X_test_m6, y_test_m6 = generate_fraud_data(
    n_samples=500, fraud_ratio=0.05,
    amount_mean_legit=180,  # DATA DRIFT: legit amounts increased
    amount_mean_fraud=350,  # fraud amounts slightly decreased (camouflage)
    time_mean_fraud=14.0,   # CONCEPT DRIFT: fraud now happens at 2 PM
    merchant_fraud_prob=[0.6, 0.3, 0.1]  # CONCEPT DRIFT: category 0 now risky
)

acc_month6 = model.score(X_test_m6, y_test_m6)
print(f"Month 6 Accuracy: {acc_month6:.3f}")
print(f"Degradation: {(acc_baseline - acc_month6) / acc_baseline * 100:.1f}%")

# Feature analysis
def feature_stats(X, y, month):
    df = pd.DataFrame(X, columns=['amount', 'time_of_day', 'merchant_cat'])
    df['is_fraud'] = y
    print(f"\n{month} - Feature Statistics:")
    print(df.groupby('is_fraud')[['amount', 'time_of_day']].mean())

feature_stats(X_test_m1, y_test_m1, "Month 1")
feature_stats(X_test_m6, y_test_m6, "Month 6")

print("\n=== DIAGNOSIS ===")
print("DATA DRIFT: Legitimate transaction amounts increased (e-commerce boom)")
print("CONCEPT DRIFT: Fraud time shifted from 2 AM → 2 PM")
print("CONCEPT DRIFT: High-risk merchant category changed from 2 → 0")
print("Result: The boundary learned in Month 1 no longer separates fraud effectively.")

## Solution 3: The Frozen Model Problem

**Answer:**

Three real-world changes and their classifications:

1. **New metro line opened in March** (Data Drift)
   - *Classification:* Data drift - P(X) changed
   - *Reasoning:* New geographic patterns emerged (new pickup/dropoff locations), but the relationship between location and demand remains the same. The input space expanded with new feature values the model never saw.

2. **Competitor launched aggressive pricing in May** (Concept Drift)
   - *Classification:* Concept drift - P(y|X) changed
   - *Reasoning:* The same conditions (time, weather, location) no longer predict the same demand level because users now have an alternative. The input-output relationship fundamentally changed.

3. **Remote work policy shift reduced commute patterns** (Both)
   - *Classification:* Both data and concept drift
   - *Reasoning:* Data drift: Distribution of ride times shifted (fewer 8 AM peaks). Concept drift: Weekday morning rush behavior now resembles weekend patterns. Both P(X) and P(y|X) moved together.

**Why this matters:** The model would need different interventions for each:
- Data drift alone → retrain on recent data
- Concept drift → might need new features or algorithm changes
- Both → comprehensive pipeline update required

## Solution 4: Comparing ML Decay to Software Decay

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

def traditional_software():
    """Tax calculator: 100% reliable until code/rules change (rare events)."""
    reliability = np.ones(12) * 100
    reliability[7] = 0   # Month 8: tax law changed but code wasn't updated
    reliability[8:] = 100  # Month 9: patched and back to 100%
    return reliability

def ml_system():
    """ML model: gradual decay as world drifts from training snapshot."""
    baseline = 95
    # Exponential decay with noise
    decay_rate = 0.08
    months = np.arange(12)
    reliability = baseline * np.exp(-decay_rate * months)
    # Add realistic noise
    reliability += np.random.normal(0, 2, 12)
    return np.clip(reliability, 60, 100)

trad = traditional_software()
ml = ml_system()

months = [f"M{i+1}" for i in range(12)]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Traditional software
ax1.plot(months, trad, marker='o', linewidth=2, markersize=8, color='steelblue')
ax1.axhline(y=90, color='green', linestyle='--', alpha=0.3, label='Acceptable threshold')
ax1.set_ylim([0, 105])
ax1.set_ylabel('Reliability (%)', fontsize=11)
ax1.set_xlabel('Month', fontsize=11)
ax1.set_title('Traditional Software (Tax Calculator)', fontsize=12, fontweight='bold')
ax1.grid(alpha=0.3)
ax1.annotate('Tax law changed\n(loud failure)', xy=(7, 0), xytext=(7, 30),
            arrowprops=dict(arrowstyle='->', color='red', lw=2),
            fontsize=10, color='red', ha='center')
ax1.legend()

# ML system
ax2.plot(months, ml, marker='s', linewidth=2, markersize=8, color='coral')
ax2.axhline(y=90, color='green', linestyle='--', alpha=0.3, label='Acceptable threshold')
ax2.axhline(y=80, color='orange', linestyle='--', alpha=0.3, label='Warning threshold')
ax2.set_ylim([0, 105])
ax2.set_ylabel('Reliability (%)', fontsize=11)
ax2.set_xlabel('Month', fontsize=11)
ax2.set_title('ML System (Fraud Detector)', fontsize=12, fontweight='bold')
ax2.grid(alpha=0.3)
ax2.annotate('Silent drift\n(gradual decay)', xy=(9, ml[9]), xytext=(9, 50),
            arrowprops=dict(arrowstyle='->', color='red', lw=2),
            fontsize=10, color='red', ha='center')
ax2.legend()

plt.tight_layout()
plt.savefig('sample_data/ml_vs_traditional_decay.png', dpi=100, bbox_inches='tight')
plt.show()

print("=== KEY INSIGHTS ===")
print(f"Traditional software:")
print(f"  - Starts at: {trad[0]:.0f}% | Ends at: {trad[-1]:.0f}%")
print(f"  - Failure mode: LOUD (crashes, errors)")
print(f"  - Months below 90%: {(trad < 90).sum()} out of 12\n")

print(f"ML system:")
print(f"  - Starts at: {ml[0]:.1f}% | Ends at: {ml[-1]:.1f}%")
print(f"  - Failure mode: SILENT (gradual accuracy loss)")
print(f"  - Months below 90%: {(ml < 90).sum()} out of 12")
print(f"  - Total degradation: {ml[0] - ml[-1]:.1f} percentage points")
print("\nMoral: Traditional software breaks loudly; ML systems smile while decaying.")

## Solution 5: ML Lifecycle Mapping

**Telecom Churn Prediction - Lifecycle Analysis:**

| Stage | Key Question | Artifact | Silent Failure |
|-------|--------------|----------|----------------|
| **1. Scope** | Is ML better than a rule ("customer called support 3x → offer discount")? | Problem definition doc + success metric (precision ≥ 0.75) | Ship an over-engineered ML solution when a rule would work better and be more maintainable |
| **2. Data** | Do we have reliable churn labels within 30 days of prediction? | Validated, versioned dataset with schema + quality checks | Train on mislabeled data (e.g., "churned" includes users who paused service temporarily) |
| **3. Model** | Does the model beat "predict majority class" baseline? | Tracked experiment runs + evaluation report (confusion matrix, feature importance) | Ship a model that barely beats random chance because no baseline comparison was made |
| **4. Deploy** | Can the CRM system call this API and act on predictions in real-time? | Production FastAPI endpoint + health checks + integration tests | API returns predictions but downstream system ignores them (integration never validated) |
| **5. Monitor** | Is the model still accurate on THIS month's data? | Dashboards showing accuracy, input drift (PSI), prediction distribution | Input distribution shifts (new pricing plan) but nobody notices until churn offers stop working |
| **6. Retrain** | Should we retrain on fresh data or fix features first? | New validated model candidate + decision log | Auto-retrain on schedule even though a feature broke (retraining bakes in the bug) |

**Cross-cutting insight:** The silent failures share a pattern—they all look "green" on surface metrics while being fundamentally broken underneath.

## Solution 6: Component Matching

**Correct Matches:**

1. **Experiment tracking** → **A** ("Which parameters produced our best model last month?")
   - Tracks params, metrics, and artifacts across training runs

2. **Data versioning** → **F** ("Did someone edit the training CSV, or is this the same data?")
   - Content-addresses datasets so changes are detectable

3. **Model registry** → **D** ("Which model version is currently in production?")
   - Tracks model versions and their deployment stages

4. **Orchestration** → **C** ("How do we run data validation before training every night at 2 AM?")
   - Schedules and chains pipeline steps

5. **Serving** → **E** ("How do applications call the model via HTTP?")
   - Exposes models as APIs with validation

6. **Monitoring** → **B** ("Is the input distribution shifting away from training data?")
   - Watches for drift, data quality issues, and performance degradation

## Solution 7: Maturity Assessment

In [ ]:
from typing import Dict

def maturity_assessment(answers: Dict[str, bool]) -> str:
    """
    Assess MLOps maturity based on 10 yes/no questions.
    
    Returns: Level 0, 1, or 2 with explanation
    """
    questions = {
        'q1_version_control': 'Is training code in version control (git)?',
        'q2_automated_pipeline': 'Is training a single automated script/pipeline (not manual notebook execution)?',
        'q3_experiment_tracking': 'Are experiments tracked with params/metrics logged automatically?',
        'q4_data_versioning': 'Are datasets versioned or content-addressed?',
        'q5_model_registry': 'Is there a registry showing which model version is in production?',
        'q6_automated_tests': 'Do automated tests run on every code change?',
        'q7_data_validation': 'Does data validation run automatically before training?',
        'q8_monitoring_dashboard': 'Is there live monitoring of model predictions/drift?',
        'q9_ci_cd': 'Do tests + deployment happen automatically on merge to main?',
        'q10_auto_retrain': 'Does retraining trigger automatically (schedule or drift)?'
    }
    
    # Level 2 indicators
    level_2_score = sum([
        answers.get('q9_ci_cd', False),
        answers.get('q10_auto_retrain', False),
        answers.get('q7_data_validation', False),
        answers.get('q6_automated_tests', False)
    ])
    
    # Level 1 indicators
    level_1_score = sum([
        answers.get('q2_automated_pipeline', False),
        answers.get('q3_experiment_tracking', False),
        answers.get('q8_monitoring_dashboard', False)
    ])
    
    # Determine level
    if level_2_score >= 3 and answers.get('q10_auto_retrain', False):
        level = 2
        desc = "CI/CD + Continuous Training"
        strengths = "Fully automated testing, deployment, and retraining"
    elif level_1_score >= 2 and answers.get('q2_automated_pipeline', False):
        level = 1
        desc = "Automated Pipeline"
        strengths = "Reproducible training, basic tracking and monitoring"
    else:
        level = 0
        desc = "Manual / Ad-hoc"
        strengths = "Flexibility and rapid experimentation"
    
    # Calculate score
    total_yes = sum(answers.values())
    
    report = f"""
    ╔══════════════════════════════════════════════════════╗
    ║         MLOps Maturity Assessment Report            ║
    ╚══════════════════════════════════════════════════════╝
    
    Maturity Level: {level}
    Classification: {desc}
    Score: {total_yes}/10 practices implemented
    
    Strengths: {strengths}
    
    Next Steps to Level {min(level + 1, 2)}:
    """
    
    if level == 0:
        report += """
    1. Convert key notebooks to automated Python scripts
    2. Add basic experiment tracking (even file-based)
    3. Version training data with hashes or DVC
    4. Create a simple monitoring dashboard
        """
    elif level == 1:
        report += """
    1. Add automated testing (data validation + model contracts)
    2. Implement CI/CD pipeline with metric gates
    3. Set up automatic retrain triggers (schedule or drift)
    4. Formalize model registry with staging/prod stages
        """
    else:
        report += """
    You're at the top! Focus on:
    1. Optimizing feedback loops (faster detection → action)
    2. A/B testing infrastructure for gradual rollouts
    3. Advanced monitoring (segment-level metrics, fairness)
    4. Sharing practices across teams
        """
    
    return report

# Assess the fictional team from the exercise
fictional_team = {
    'q1_version_control': True,        # Git repo exists
    'q2_automated_pipeline': False,    # Manual notebook execution
    'q3_experiment_tracking': False,   # No systematic tracking
    'q4_data_versioning': False,       # Just a 'models/' folder with timestamps
    'q5_model_registry': False,        # No formal registry
    'q6_automated_tests': False,       # No tests mentioned
    'q7_data_validation': False,       # No validation mentioned
    'q8_monitoring_dashboard': True,   # Monthly accuracy dashboard exists
    'q9_ci_cd': False,                 # Manual deployment (copy .pkl)
    'q10_auto_retrain': False          # Retraining when "someone remembers"
}

print(maturity_assessment(fictional_team))

print("\n" + "="*60)
print("ANALYSIS OF THE FICTIONAL TEAM:")
print("="*60)
print("""
This team is solidly at Level 0 (Manual/Ad-hoc) because:

Positive signals:
- Code is in version control (foundation for future automation)
- Some monitoring exists (monthly dashboard)

Critical gaps:
- No automated pipeline (manual notebook execution)
- No experiment tracking (can't compare past runs)
- No data versioning ("which data trained this model?" is unanswerable)
- Manual deployment (high risk, slow, error-prone)
- Reactive retraining ("when someone remembers")

The monthly dashboard is TOO SLOW for real issues. Drift could cause
problems for weeks before it's detected.

Priority fixes:
1. Convert the notebook to a script with config files
2. Add basic experiment logging (even JSON files)
3. Version the training data (start with file hashes)
4. Schedule regular retraining (weekly cron job)
""")

## Solution 8: DevOps vs MLOps Comparison Table

| Dimension | Traditional Software (DevOps) | ML System (MLOps) |
|-----------|-------------------------------|-------------------|
| **Core Artifact** | Code (application logic) | Code + Data + Trained weights |
| **Versioning** | Git for code | Git for code + DVC/hash for data + registry for models |
| **Testing** | Unit tests with exact assertions | Unit tests + data validation + statistical metric gates |
| **Monitoring** | Errors, latency, resource usage | Those + input/output distributions + drift + business metrics |
| **Deploy Trigger** | Code merged to main | Code merge OR scheduled retrain OR drift detection |
| **Failure Mode** | Loud crashes, stack traces, alerts | Silent degradation over weeks/months |
| **Rollback Complexity** | Redeploy previous code version | Rollback code + model + potentially data pipeline |
| **Reproducibility** | Same code → same behavior | Same code + data + seeds + environment → same model |
| **Performance Definition** | Latency, throughput, uptime | Those + prediction accuracy, precision, recall |
| **Dependencies** | Code libraries (npm, pip) | Code libraries + data pipelines + training infrastructure |
| **Team Composition** | Developers + Ops/SRE | Those + Data Scientists + Data Engineers + Domain Experts |
| **Typical SLA** | 99.9% uptime | Uptime + minimum accuracy threshold |
| **Decay Pattern** | Stable until code/config changes | Gradual decay as world drifts from training snapshot |
| **CI/CD Scope** | Build → Test → Deploy | Build → Test → Train → Evaluate → Deploy (+ continuous retraining) |
| **Debugging Challenge** | Find the breaking code change | Isolate if it's code, data, drift, or feature engineering |
| **Cost Drivers** | Compute, storage | Those + training compute + annotation labor + retraining frequency |

## Solution 9: Postmortem Analysis

## POSTMORTEM: E-Commerce Recommendation Model Degradation

**Incident Summary:**  
Click-through rate (CTR) dropped 40% from January baseline to June, with no error logs or alerts. Model continued serving predictions with high confidence.

---

### 1. Root Cause Analysis

**Primary Root Cause:**  
The recommendation model was trained exclusively on clothing purchase patterns and had no mechanism to handle or recommend electronics products added in March.

**Contributing Factors:**

1. **March: New product category (Electronics)**
   - Model trained only on clothing attributes (color, size, style)
   - Electronics require different features (specs, compatibility)
   - Model likely returned random/low-quality electronics recommendations
   - **Classification:** Data drift (new category = P(X) changed) + model inadequacy

2. **April: Mobile app redesign**
   - Changed user browsing patterns and interaction signals
   - Model relied on old UX patterns (e.g., "time on product page")
   - New swipe-based interface invalidated behavioral features
   - **Classification:** Concept drift - same user intent, different signals (P(y|X) changed)

3. **May: Competitor launch**
   - Users comparison-shopping across platforms
   - Reduced on-site engagement even for relevant recommendations
   - **Classification:** Concept drift - external market dynamics changed user behavior

---

### 2. Why This Went Undetected

- **No input monitoring:** Electronics products appeared in feature vectors, but no alerts on new category values
- **No output monitoring:** Distribution of recommended products not tracked
- **No A/B baseline:** No control group receiving random/popularity-based recommendations to compare against
- **Wrong metric:** System tracked error rate (always 0%) instead of business metric (CTR)
- **Delayed feedback:** CTR aggregated weekly; individual bad recommendations went unnoticed

---

### 3. What Monitoring Would Have Caught This

**Input Monitoring (would have alerted in March):**
- PSI on categorical features would spike when "electronics" appeared
- Alert: "Unknown category value in 15% of requests"

**Output Monitoring (would have alerted in March-April):**
- Recommendation diversity metrics (started recommending same items repeatedly)
- Confidence score distribution (likely dropped for electronics)
- Category distribution (all recs still clothing despite 30% electronics requests)

**Business Metric Monitoring (would have alerted in April):**
- CTR tracking per product category
- Daily CTR trend (not weekly aggregates)
- Alert threshold: "CTR dropped >10% for 3 consecutive days"

**Feature Importance Drift (would have alerted in April):**
- Track which features model relies on
- Alert when key features (time-on-page) become unavailable/different post-redesign

---

### 4. Action Items

| Action | Owner | Deadline | Priority |
|--------|-------|----------|----------|
| **Immediate (Week 1)** | | | |
| Retrain model including electronics data | Data Science | Week 1 | P0 |
| Implement input PSI monitoring on category field | ML Eng | Week 1 | P0 |
| Add daily CTR dashboard by category | Analytics | Week 1 | P0 |
| **Short-term (Month 1)** | | | |
| Redesign features for new mobile UX | DS + Product | Month 1 | P0 |
| Implement output monitoring (diversity, confidence) | ML Eng | Month 1 | P1 |
| Set up automated alerts: CTR <-10% for 2 days → page on-call | MLOps | Month 1 | P0 |
| Create A/B test framework with popularity baseline | ML Eng | Month 1 | P1 |
| **Medium-term (Quarter)** | | | |
| Implement shadow mode for model updates | ML Eng | Q2 | P1 |
| Build automatic retraining pipeline (monthly) | MLOps | Q2 | P1 |
| Add feature drift monitoring (correlation shifts) | ML Eng | Q2 | P2 |
| Document runbook: "Model performance degraded" | Team Lead | Q2 | P1 |
| **Process Changes** | | | |
| Require product changes to notify ML team 2 weeks advance | Product Mgr | Immediate | P0 |
| Monthly model health review meeting | Team Lead | Immediate | P1 |
| Add "recommendation quality" to product launch checklist | Product Mgr | Month 1 | P1 |

---

### 5. Lessons Learned

1. **ML models are fragile to product changes:** UX redesigns aren't just frontend changes—they're data distribution shifts
2. **Error rate ≠ model health:** 0% errors while serving bad recommendations
3. **Business metrics > model metrics:** Accuracy in dev doesn't guarantee CTR in prod
4. **Monitoring must be proactive:** Weekly reports are postmortems, not monitoring
5. **Cross-team communication is infrastructure:** Product/ML sync is not optional

---

**Estimated Impact:**  
- Revenue loss: ~$240K (40% CTR drop over 3 months)
- Prevention cost: ~$15K (monitoring + alerts + retrain automation)
- ROI of prevention: 16x

**Sign-off:**  
ML Team Lead: _______________  Date: _______________  
Product Manager: _______________  Date: _______________  

## Solution 10: Simulate the Feedback Loop

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

np.random.seed(42)

class MLLifecycleSimulator:
    def __init__(self, retrain_threshold=0.85, drift_rate=0.03):
        self.model = None
        self.retrain_threshold = retrain_threshold
        self.drift_rate = drift_rate
        self.history = []
        self.retrain_count = 0
        
    def generate_data(self, n_samples=1000, time_period=0):
        """Generate data that drifts over time."""
        # Concept shifts: the decision boundary rotates over time
        drift_factor = time_period * self.drift_rate
        
        # Original pattern: y = 1 if x1 > 0.5
        # Drifted pattern: boundary shifts
        X = np.random.randn(n_samples, 2)
        
        # Decision boundary shifts over time
        boundary = 0.5 - drift_factor
        y = (X[:, 0] + drift_factor * X[:, 1] > boundary).astype(int)
        
        return X, y
    
    def train(self, time_period):
        """Train or retrain the model."""
        print(f"\n🔄 [Period {time_period}] Training model...")
        X_train, y_train = self.generate_data(n_samples=2000, time_period=time_period)
        self.model = LogisticRegression(max_iter=1000)
        self.model.fit(X_train, y_train)
        self.retrain_count += 1
        print(f"✓ Model trained (retrain #{self.retrain_count})")
        return time_period  # Model now reflects this period's data
    
    def serve_and_monitor(self, current_period, model_trained_period):
        """Serve predictions and monitor performance."""
        # Generate current period's data (drifted from model training time)
        X_current, y_current = self.generate_data(n_samples=500, time_period=current_period)
        
        # Serve predictions
        y_pred = self.model.predict(X_current)
        accuracy = accuracy_score(y_current, y_pred)
        
        # Calculate drift (how far current period is from model training period)
        drift_amount = abs(current_period - model_trained_period) * self.drift_rate
        
        status = "🟢 HEALTHY" if accuracy >= self.retrain_threshold else "🔴 DEGRADED"
        
        record = {
            'period': current_period,
            'accuracy': accuracy,
            'drift': drift_amount,
            'model_age': current_period - model_trained_period,
            'status': status
        }
        
        self.history.append(record)
        
        print(f"📊 [Period {current_period}] Acc: {accuracy:.3f} | "
              f"Drift: {drift_amount:.3f} | Model age: {record['model_age']} | {status}")
        
        return accuracy
    
    def run_lifecycle(self, n_periods=20):
        """Run the full ML lifecycle loop."""
        print("=" * 70)
        print("🚀 Starting ML Lifecycle Simulation")
        print(f"Retrain threshold: {self.retrain_threshold}")
        print(f"Drift rate: {self.drift_rate} per period")
        print("=" * 70)
        
        # Initial training
        model_trained_period = self.train(time_period=0)
        
        # Serve and monitor loop
        for period in range(n_periods):
            accuracy = self.serve_and_monitor(period, model_trained_period)
            
            # Decision: retrain if degraded
            if accuracy < self.retrain_threshold:
                print(f"⚠️  Accuracy {accuracy:.3f} < threshold {self.retrain_threshold}")
                print(f"🔁 Triggering retrain on fresh data...")
                model_trained_period = self.train(time_period=period)
        
        print("\n" + "=" * 70)
        print("📈 Lifecycle Summary")
        print("=" * 70)
        print(f"Total periods: {n_periods}")
        print(f"Retraining events: {self.retrain_count - 1}")  # -1 for initial training
        
        import pandas as pd
        df = pd.DataFrame(self.history)
        print(f"\nAccuracy: min={df['accuracy'].min():.3f}, "
              f"max={df['accuracy'].max():.3f}, "
              f"mean={df['accuracy'].mean():.3f}")
        print(f"Periods below threshold: {(df['accuracy'] < self.retrain_threshold).sum()}")
        
        return df

# Run simulation
simulator = MLLifecycleSimulator(retrain_threshold=0.85, drift_rate=0.03)
history_df = simulator.run_lifecycle(n_periods=20)

# Visualize
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))

# Accuracy over time
ax1.plot(history_df['period'], history_df['accuracy'], 
         marker='o', linewidth=2, markersize=6, label='Accuracy')
ax1.axhline(y=simulator.retrain_threshold, color='red', 
            linestyle='--', label='Retrain threshold', linewidth=2)

# Mark retraining events
retrain_periods = [0] + [i for i in range(1, len(history_df)) 
                         if history_df.iloc[i]['model_age'] == 0]
for period in retrain_periods[1:]:  # Skip initial training
    ax1.axvline(x=period, color='green', alpha=0.3, linestyle=':', linewidth=2)
    ax1.text(period, 0.80, '🔄', fontsize=16, ha='center')

ax1.set_ylabel('Accuracy', fontsize=11)
ax1.set_xlabel('Time Period', fontsize=11)
ax1.set_title('ML Lifecycle: Accuracy Over Time with Auto-Retraining', 
              fontsize=12, fontweight='bold')
ax1.legend()
ax1.grid(alpha=0.3)
ax1.set_ylim([0.75, 1.0])

# Model age over time
ax2.plot(history_df['period'], history_df['model_age'], 
         marker='s', linewidth=2, markersize=6, color='coral', label='Model Age')
ax2.set_ylabel('Model Age (periods)', fontsize=11)
ax2.set_xlabel('Time Period', fontsize=11)
ax2.set_title('Model Age: Resets After Each Retrain', fontsize=12, fontweight='bold')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('sample_data/ml_lifecycle_simulation.png', dpi=100, bbox_inches='tight')
plt.show()

print("\n💡 KEY INSIGHT:")
print("The feedback loop prevents catastrophic decay. Without auto-retraining,")
print("accuracy would drop to ~0.60 by period 20. With it, accuracy stays above 0.85.")

## Solution 11: Role Assignment

**Role Assignments:**

1. **Alex (ML background, strong Python)** → **Data Scientist + ML Engineer**
   - *Justification:* ML background makes them suitable for modeling work, and Python skills enable them to productionize their own models initially.
   - *Responsibilities:* Feature engineering, model training, evaluation, and initial API development.

2. **Jordan (data engineering, SQL expert)** → **Data Engineer**
   - *Justification:* Perfect fit for the data pipeline role.
   - *Responsibilities:* Build data ingestion pipelines, ensure data quality, create feature stores, manage data versioning.

3. **Sam (backend dev, infrastructure experience)** → **MLOps Engineer**
   - *Justification:* Infrastructure expertise is critical for deployment, monitoring, and CI/CD.
   - *Responsibilities:* Set up model serving infrastructure, implement monitoring, create CI/CD pipelines, manage deployments.

4. **Riley (domain expert, 10 years in fraud prevention)** → **Domain Expert + Product Advisor**
   - *Justification:* Domain knowledge is invaluable for feature engineering, model interpretation, and setting appropriate thresholds.
   - *Responsibilities:* Define fraud patterns, validate model behavior, help with labeling strategy, review model decisions for false positives/negatives.

5. **Morgan (product manager)** → **Product Manager**
   - *Justification:* Obvious PM fit.
   - *Responsibilities:* Define success metrics, prioritize features, coordinate between teams, own the product roadmap, translate business requirements.

**Gaps Identified:**

1. **No dedicated ML Engineer** (Alex is split between DS and MLE roles)
   - *Risk:* Models might not be production-ready; technical debt accumulates
   - *Mitigation:* Train Alex in software engineering best practices; consider hiring a dedicated MLE as the team scales

2. **No specialized MLOps expertise** (Sam has general infra, but not ML-specific)
   - *Risk:* Missing ML-specific monitoring (drift detection, model versioning)
   - *Mitigation:* Invest in Sam's MLOps training; bring in a consultant for initial setup

3. **No security specialist**
   - *Risk:* Fraud detection systems are high-security; need expertise in secure ML
   - *Mitigation:* Engage security team for review; Riley's fraud expertise partially covers this

**Recommended Hiring Priority:**
1. **Next hire:** ML Engineer (to free Alex to focus on modeling)
2. **After scaling:** Second Data Scientist (for experimentation velocity)
3. **Long-term:** Security Engineer with ML focus

## Solution 12: Challenge - Build a Mini MLOps Dashboard

In [ ]:
from datetime import datetime, timedelta
from dataclasses import dataclass
from typing import List, Dict
import numpy as np

@dataclass
class FeatureDriftStatus:
    feature_name: str
    psi: float
    status: str  # 'stable', 'warning', 'alert'
    
    def get_icon(self):
        return {'stable': '🟢', 'warning': '🟡', 'alert': '🔴'}[self.status]

@dataclass
class ModelDashboard:
    model_version: str
    deploy_date: datetime
    last_retrain: datetime
    accuracy_history: List[float]  # last 7 days
    feature_drift: List[FeatureDriftStatus]
    
    def get_overall_status(self) -> tuple:
        """Return (status, icon) based on multiple signals."""
        recent_acc = np.mean(self.accuracy_history[-3:])  # last 3 days
        has_drift_alert = any(f.status == 'alert' for f in self.feature_drift)
        
        if recent_acc < 0.85 or has_drift_alert:
            return ('CRITICAL', '🔴')
        elif recent_acc < 0.90 or any(f.status == 'warning' for f in self.feature_drift):
            return ('WARNING', '🟡')
        else:
            return ('HEALTHY', '🟢')
    
    def days_since_retrain(self) -> int:
        return (datetime.now() - self.last_retrain).days
    
    def render(self):
        """Render text-based dashboard."""
        status, icon = self.get_overall_status()
        acc_trend = self._get_trend_arrow()
        
        dashboard = f"""
╔═══════════════════════════════════════════════════════════════════════════╗
║                       🤖  ML MODEL HEALTH DASHBOARD                       ║
╠═══════════════════════════════════════════════════════════════════════════╣
║                                                                           ║
║  Overall Status: {icon} {status:<20}  Last Update: {datetime.now().strftime('%Y-%m-%d %H:%M')}  ║
║                                                                           ║
╠═══════════════════════════════════════════════════════════════════════════╣
║  MODEL INFO                                                               ║
╠═══════════════════════════════════════════════════════════════════════════╣
║  Version:           {self.model_version:<30}                              ║
║  Deployed:          {self.deploy_date.strftime('%Y-%m-%d'):<30} ({(datetime.now() - self.deploy_date).days} days ago)    ║
║  Last Retrain:      {self.last_retrain.strftime('%Y-%m-%d'):<30} ({self.days_since_retrain()} days ago)    ║
║                                                                           ║
╠═══════════════════════════════════════════════════════════════════════════╣
║  ACCURACY TREND (Last 7 Days) {acc_trend}                                      ║
╠═══════════════════════════════════════════════════════════════════════════╣
"""
        # Accuracy sparkline
        sparkline = self._create_sparkline(self.accuracy_history)
        dashboard += f"║  {sparkline:<73}║\n"
        dashboard += f"║  Current: {self.accuracy_history[-1]:.3f}  |  7-day avg: {np.mean(self.accuracy_history):.3f}  |  Min: {min(self.accuracy_history):.3f}  |  Max: {max(self.accuracy_history):.3f}  ║\n"
        
        # Feature drift
        dashboard += """║                                                                           ║
╠═══════════════════════════════════════════════════════════════════════════╣
║  FEATURE DRIFT STATUS (PSI)                                               ║
╠═══════════════════════════════════════════════════════════════════════════╣
"""
        for feat in self.feature_drift:
            dashboard += f"║  {feat.get_icon()} {feat.feature_name:<25} PSI: {feat.psi:>6.3f}  ({feat.status.upper():<8})   ║\n"
        
        # Recommendations
        dashboard += """║                                                                           ║
╠═══════════════════════════════════════════════════════════════════════════╣
║  RECOMMENDED ACTIONS                                                      ║
╠═══════════════════════════════════════════════════════════════════════════╣
"""
        recommendations = self._get_recommendations()
        for rec in recommendations:
            dashboard += f"║  • {rec:<71}║\n"
        
        dashboard += """║                                                                           ║
╚═══════════════════════════════════════════════════════════════════════════╝
"""
        return dashboard
    
    def _get_trend_arrow(self) -> str:
        if len(self.accuracy_history) < 2:
            return '→'
        recent_trend = self.accuracy_history[-1] - self.accuracy_history[-3]
        if recent_trend > 0.01:
            return '📈'
        elif recent_trend < -0.01:
            return '📉'
        return '→'
    
    def _create_sparkline(self, values: List[float]) -> str:
        """Create ASCII sparkline."""
        chars = ['▁', '▂', '▃', '▄', '▅', '▆', '▇', '█']
        min_val, max_val = min(values), max(values)
        if max_val == min_val:
            return chars[4] * len(values) + "  (stable)"
        
        normalized = [(v - min_val) / (max_val - min_val) for v in values]
        sparkline = ''.join(chars[min(int(n * len(chars)), len(chars) - 1)] for n in normalized)
        return f"  {sparkline}  " + " ".join([f"{v:.3f}" for v in values])
    
    def _get_recommendations(self) -> List[str]:
        recs = []
        status, _ = self.get_overall_status()
        
        if status == 'CRITICAL':
            recs.append("🚨 URGENT: Trigger immediate retrain or rollback to previous version")
        
        if any(f.status == 'alert' for f in self.feature_drift):
            alert_features = [f.feature_name for f in self.feature_drift if f.status == 'alert']
            recs.append(f"Investigate drift in: {', '.join(alert_features)}")
        
        if self.days_since_retrain() > 30:
            recs.append(f"Model is {self.days_since_retrain()} days old - schedule retrain")
        
        if np.mean(self.accuracy_history[-3:]) < 0.90:
            recs.append("Recent accuracy below 0.90 - review recent data quality")
        
        if not recs:
            recs.append("✓ All systems nominal - continue monitoring")
        
        return recs

# Simulate a dashboard with various states
def create_sample_dashboard(scenario='healthy'):
    if scenario == 'healthy':
        accuracy = [0.945, 0.947, 0.948, 0.946, 0.949, 0.947, 0.950]
        drift = [
            FeatureDriftStatus('transaction_amount', 0.06, 'stable'),
            FeatureDriftStatus('customer_age', 0.08, 'stable'),
            FeatureDriftStatus('merchant_category', 0.12, 'warning')
        ]
        last_retrain = datetime.now() - timedelta(days=7)
    
    elif scenario == 'warning':
        accuracy = [0.920, 0.915, 0.910, 0.905, 0.900, 0.895, 0.890]
        drift = [
            FeatureDriftStatus('transaction_amount', 0.15, 'warning'),
            FeatureDriftStatus('customer_age', 0.18, 'warning'),
            FeatureDriftStatus('merchant_category', 0.08, 'stable')
        ]
        last_retrain = datetime.now() - timedelta(days=21)
    
    else:  # critical
        accuracy = [0.890, 0.870, 0.850, 0.830, 0.820, 0.810, 0.800]
        drift = [
            FeatureDriftStatus('transaction_amount', 0.28, 'alert'),
            FeatureDriftStatus('customer_age', 0.31, 'alert'),
            FeatureDriftStatus('merchant_category', 0.22, 'warning')
        ]
        last_retrain = datetime.now() - timedelta(days=45)
    
    return ModelDashboard(
        model_version='fraud-detector-v2.3.1',
        deploy_date=datetime.now() - timedelta(days=60),
        last_retrain=last_retrain,
        accuracy_history=accuracy,
        feature_drift=drift
    )

# Demo all three scenarios
for scenario in ['healthy', 'warning', 'critical']:
    print(f"\n\n{'='*80}")
    print(f"SCENARIO: {scenario.upper()}")
    print('='*80)
    dashboard = create_sample_dashboard(scenario)
    print(dashboard.render())

print("\n💡 This dashboard combines multiple signals (accuracy, drift, model age)")
print("   into actionable recommendations. In production, this would update in real-time.")